# Plant leaves disease detection using CNN - Phase 2 GPU baseline

Run this notebook on Kaggle. Turn on the GPU accelerator and attach the prepared PlantVillage dataset. The notebook uses the dataset at /kaggle/input/datasets/shubhamkumar12864/plantvillage/. It searches other Kaggle input folders if that path is not available.

The notebook uses the saved train, validation, and test manifests. It trains a SmallCNN on the training images. It checks validation macro-F1 after each epoch. It saves the best checkpoint. It evaluates the locked test set once after training.

With two T4 GPUs, the notebook uses DataParallel and a global batch size of 64. Each GPU receives 32 images per batch. With one T4 GPU, it uses a batch size of 32. It uses mixed precision and NUM_WORKERS = 0. This avoids the DataLoader worker errors seen in an earlier run. Two GPUs can improve speed. They do not guarantee twice the speed.

The notebook resizes and normalizes images as each batch loads. It applies random augmentation to training images only. It saves the checkpoint, metrics, reports, confusion matrix, training history, chart, configuration, and a ZIP file under /kaggle/working/.


In [ ]:
from pathlib import Path
import json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from tqdm.auto import tqdm

SEED = 42
GPU_COUNT = torch.cuda.device_count()
GPUS_USED = min(GPU_COUNT, 2)
BATCH_SIZE = 64 if GPUS_USED == 2 else 32  # 32 images per T4 with two GPUs.
EPOCHS = 12
PATIENCE = 3
IMAGE_SIZE = 224
NUM_WORKERS = 0  # Use the main process. This avoids Kaggle DataLoader worker errors.
PREFETCH_FACTOR = 1
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True  # Input size stays fixed at 224x224.
    gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
else:
    gpu_names = []
print('Device:', device, '| visible GPUs:', gpu_names or 'none', '| GPUs used:', GPUS_USED, '| global batch:', BATCH_SIZE)
if device.type != 'cuda':
    raise RuntimeError('Enable a Kaggle GPU accelerator for full training; CPU is not recommended for this image workload.')
if len(gpu_names) > 1:
    print('Multi-GPU enabled with DataParallel; batch is split across the first two GPUs (32 images each).')

# Kaggle input path provided for the attached PlantVillage dataset.
preferred_root = Path('/kaggle/input/datasets/shubhamkumar12864/plantvillage')
if (preferred_root / 'data' / 'manifests' / 'plantvillage_development.csv').is_file():
    project_root = preferred_root
else:
    # Kaggle sometimes mounts the dataset directly by slug; search the same folder structure.
    input_root = Path('/kaggle/input')
    candidates = []
    for pattern in (
        '*/data/manifests/plantvillage_development.csv',
        '*/*/data/manifests/plantvillage_development.csv',
        '*/*/*/data/manifests/plantvillage_development.csv',
        '*/*/*/*/data/manifests/plantvillage_development.csv',
    ):
        candidates.extend(input_root.glob(pattern))
    if not candidates:
        raise FileNotFoundError(
            'Could not find data/manifests/plantvillage_development.csv. '
            'Check that the PlantVillage Kaggle Dataset is attached as an input.'
        )
    # The manifest is at <dataset-root>/data/manifests/<file>.
    project_root = candidates[0].parents[2]
    print('Using discovered Kaggle dataset path:', project_root)

manifest_dir = project_root / 'data' / 'manifests'
image_root = project_root / 'data' / 'raw' / 'PlantVillage'
dev = pd.read_csv(manifest_dir / 'plantvillage_development.csv')
test = pd.read_csv(manifest_dir / 'plantvillage_test.csv')
train_df = dev[dev['split'] == 'train'].reset_index(drop=True)
val_df = dev[dev['split'] == 'validation'].reset_index(drop=True)
assert len(train_df) and len(val_df) and len(test), 'Expected non-empty train, validation, and test partitions.'
classes = sorted(train_df['label'].unique())
label_to_idx = {name: i for i, name in enumerate(classes)}
assert set(val_df['label']) == set(classes) and set(test['label']) == set(classes), 'Train, validation, and test must contain the same 38 classes.'
print(f'Project: {project_root} | train={len(train_df):,}, val={len(val_df):,}, test={len(test):,}, classes={len(classes)}')

In [ ]:
class LeafImages(Dataset):
    def __init__(self, frame, training=False):
        # Convert Pandas columns to plain lists once; avoid per-image iloc overhead.
        frame = frame.reset_index(drop=True)
        split_folders = ['test' if split == 'test' else 'train' for split in frame['split'].tolist()]
        self.paths = [image_root / folder / rel for folder, rel in zip(split_folders, frame['relative_path'].tolist())]
        self.labels = [label_to_idx[label] for label in frame['label'].tolist()]
        ops = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
        if training:
            ops += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(12), transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.10)]
        ops += [transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)]
        self.transform = transforms.Compose(ops)
    def __len__(self): return len(self.paths)
    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert('RGB')
        return self.transform(image), self.labels[index]

pin_memory = device.type == 'cuda'
train_loader = DataLoader(LeafImages(train_df, True), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin_memory, persistent_workers=False, prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None)
val_loader = DataLoader(LeafImages(val_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory, persistent_workers=False, prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None)
test_loader = DataLoader(LeafImages(test), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory, persistent_workers=False, prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None)

class SmallCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        blocks=[]; in_ch=3
        for out_ch in (32, 64, 128, 256):
            blocks += [nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            in_ch=out_ch
        self.features=nn.Sequential(*blocks)
        self.head=nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.30), nn.Linear(256, n_classes))
    def forward(self, x): return self.head(self.features(x))

base_model = SmallCNN(len(classes)).to(device)
if device.type == 'cuda':
    base_model = base_model.to(memory_format=torch.channels_last)
model = nn.DataParallel(base_model, device_ids=list(range(GPUS_USED))) if GPUS_USED > 1 else base_model
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
optimizer = torch.optim.AdamW(base_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

In [ ]:
@torch.no_grad()
def predict(loader, description):
    model.eval()
    ys, ps = [], []
    progress = tqdm(loader, desc=description, leave=False)
    for x, y in progress:
        x = x.to(device, non_blocking=True)
        if device.type == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(x)
        ps.extend(logits.argmax(1).cpu().tolist())
        ys.extend(y.tolist())
    return np.asarray(ys), np.asarray(ps)

best_f1 = -1.0
stale = 0
output_dir = Path("/kaggle/working/phase2_baseline")
output_dir.mkdir(parents=True, exist_ok=True)

if device.type == "cuda":
    for gpu_id in range(GPUS_USED):
        torch.cuda.reset_peak_memory_stats(gpu_id)

config = {
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "epochs_max": EPOCHS,
    "patience": PATIENCE,
    "image_size": IMAGE_SIZE,
    "num_workers": NUM_WORKERS,
    "prefetch_factor": PREFETCH_FACTOR,
    "mixed_precision": True,
    "architecture": "SmallCNN channels 32-64-128-256",
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "loss": "CrossEntropyLoss",
    "normalization_mean": [0.5, 0.5, 0.5],
    "normalization_std": [0.5, 0.5, 0.5],
    "train_augmentation": ["horizontal_flip", "rotation_degrees_12", "color_jitter_0.12"],
    "device": str(device),
    "visible_gpus": gpu_names,
    "gpu_count_used": GPUS_USED,
    "parallel_mode": "DataParallel" if GPUS_USED > 1 else "single_gpu",
    "torch_version": torch.__version__,
    "classes": classes,
    "train_images": len(train_df),
    "validation_images": len(val_df),
    "test_images": len(test),
}
(output_dir / "experiment_config.json").write_text(json.dumps(config, indent=2))

best_path = output_dir / "plantvillage_smallcnn_best.pt"
history = []
started = time.time()

for epoch in range(1, EPOCHS + 1):
    print(f"\nStarting epoch {epoch}/{EPOCHS}", flush=True)
    model.train()
    losses = []
    train_progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS} - train",
        leave=True,
    )

    for x, y in train_progress:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        if device.type == "cuda":
            x = x.contiguous(memory_format=torch.channels_last)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
        train_progress.set_postfix(loss=f"{loss.item():.4f}")

    yv, pv = predict(val_loader, f"Epoch {epoch}/{EPOCHS} - validation")
    val_f1 = f1_score(yv, pv, average="macro", zero_division=0)
    val_precision = precision_score(yv, pv, average="macro", zero_division=0)
    val_recall = recall_score(yv, pv, average="macro", zero_division=0)

    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(losses)),
        "val_macro_f1": float(val_f1),
        "val_macro_precision": float(val_precision),
        "val_macro_recall": float(val_recall),
        "elapsed_minutes": round((time.time() - started) / 60, 2),
    }
    history.append(row)
    print(row, flush=True)

    if val_f1 > best_f1:
        best_f1 = val_f1
        stale = 0
        torch.save(
            {
                "model_state_dict": base_model.state_dict(),
                "classes": classes,
                "image_size": IMAGE_SIZE,
                "seed": SEED,
                "epoch": epoch,
                "val_macro_f1": best_f1,
            },
            best_path,
        )
    else:
        stale += 1
        if stale >= PATIENCE:
            print("Early stopping", flush=True)
            break

pd.DataFrame(history).to_csv(output_dir / "training_history.csv", index=False)
config["training_wall_minutes"] = round((time.time() - started) / 60, 2)
config["peak_gpu_memory_mb"] = [
    round(torch.cuda.max_memory_allocated(i) / 1024**2, 1)
    for i in range(GPUS_USED)
]
(output_dir / "experiment_config.json").write_text(json.dumps(config, indent=2))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot([row["epoch"] for row in history], [row["train_loss"] for row in history], marker="o")
ax[0].set(title="Training loss", xlabel="Epoch", ylabel="Loss")
ax[1].plot([row["epoch"] for row in history], [row["val_macro_f1"] for row in history], marker="o")
ax[1].set(title="Validation macro-F1", xlabel="Epoch", ylabel="Macro-F1")
fig.tight_layout()
fig.savefig(output_dir / "training_curves.png", dpi=160)
plt.close(fig)

print("Best validation macro-F1:", best_f1, "| minutes:", config["training_wall_minutes"])


In [ ]:
# Locked test evaluation: run once after selecting the checkpoint using validation only.
checkpoint=torch.load(best_path,map_location=device,weights_only=False)
base_model.load_state_dict(checkpoint['model_state_dict'])
yt,pt=predict(test_loader, "Locked test evaluation")
metrics={'accuracy':float(accuracy_score(yt,pt)),'balanced_accuracy':float(balanced_accuracy_score(yt,pt)),'macro_f1':float(f1_score(yt,pt,average='macro',zero_division=0)),'selected_epoch':int(checkpoint['epoch']),'validation_macro_f1':float(checkpoint['val_macro_f1']),'test_images':int(len(yt)),'classes':classes,'gpu_count_used':GPUS_USED,'parallel_mode':'DataParallel' if GPUS_USED>1 else 'single_gpu','peak_gpu_memory_mb':[round(torch.cuda.max_memory_allocated(i)/1024**2,1) for i in range(GPUS_USED)]}
(output_dir/'locked_test_metrics.json').write_text(json.dumps(metrics,indent=2))
report=classification_report(yt,pt,target_names=classes,output_dict=True,zero_division=0)
pd.DataFrame(report).T.to_csv(output_dir/'locked_test_classification_report.csv')
pd.DataFrame(confusion_matrix(yt,pt),index=classes,columns=classes).to_csv(output_dir/'locked_test_confusion_matrix.csv')
print(json.dumps(metrics,indent=2))
print('Saved artifacts:',list(output_dir.iterdir()))
# One downloadable bundle containing the model and all run reports.
import shutil
artifact_zip = shutil.make_archive('/kaggle/working/phase2_baseline_artifacts', 'zip', root_dir=output_dir)
print('Downloadable bundle:', artifact_zip)

## What this notebook does

1. It loads the saved train, validation, and test manifests. It does not create new splits.
2. It resizes and normalizes images as batches load. It augments training images only.
3. It trains the CNN. It shows batch progress for each epoch.
4. It checks validation macro-F1, precision, and recall after each epoch. It saves the checkpoint with the best validation macro-F1.
5. It evaluates the locked test set after it selects the checkpoint.
6. It saves the model, configuration, training history, chart, test scores, class report, confusion matrix, and ZIP bundle.

This notebook trains a Phase 2 baseline. It does not train on PlantDoc. It does not compare several model architectures. It does not claim field performance. Use PlantDoc later for a separate external evaluation. Review and record all class mappings and exclusions before you report those results.
